# Emotion Classification: Inference
**Group 3, Section 7MA1**

[Open in Colab](https://colab.research.google.com/github/Ephab/NLP-Project/blob/main/notebooks/Emotions_inference.ipynb)

This notebook uses the models we already trained in `Emotions_classification.ipynb`, so there is no training here.
The trained models are downloaded from the project's GitHub release
(https://github.com/Ephab/NLP-Project/releases/tag/models-v1), so nothing needs to be in Google Drive.

Runs in Google Colab or local Jupyter. A GPU is not needed: RoBERTa runs fine on CPU for single sentences.
The first run downloads about 460 MB of model files (about a minute in Colab); later runs reuse them.

## 1. Setup

In [1]:
import sys, subprocess

# The TF-IDF model was saved with scikit-learn 1.6.1 (the version used for training in Colab).
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn==1.6.1'], check=True)

import json, pathlib, urllib.request, zipfile, warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # harmless tqdm notice in plain Jupyter
import joblib, sklearn, torch, transformers
from sklearn.exceptions import InconsistentVersionWarning
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# A newer scikit-learn can still load the 1.6.1 model; it only prints a version warning.
warnings.filterwarnings('ignore', category=InconsistentVersionWarning)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("sklearn     ", sklearn.__version__)
print("device      ", device)

torch        2.13.0
transformers 5.14.0
sklearn      1.8.0
device       cpu


## 2. Download the trained models
The files are saved in `saved_models/` next to this notebook (a relative path). If they are already there, the download is skipped.

In [2]:
MODELS_URL = 'https://github.com/Ephab/NLP-Project/releases/download/models-v1/saved_models.zip'
SAVE_DIR = pathlib.Path('saved_models')

if not (SAVE_DIR / 'info.json').exists():
    print("Downloading trained models (about 460 MB) from GitHub ...")
    urllib.request.urlretrieve(MODELS_URL, 'saved_models.zip')
    with zipfile.ZipFile('saved_models.zip') as z:
        z.extractall('.')
    pathlib.Path('saved_models.zip').unlink()   # keep only the extracted files

print("Model files:", sorted(p.name for p in SAVE_DIR.iterdir()))

Model files: ['info.json', 'roberta_finetuned', 'tfidf_logreg.joblib']


## 3. Load the saved models

In [3]:
# labels and settings used during training
info = json.load(open(SAVE_DIR / 'info.json'))
EKMAN      = info['labels']        # ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']
THRESHOLD  = info['threshold']     # 0.5
MAX_LENGTH = info['max_length']    # 48 tokens

# TF-IDF vectorizer + logistic regression (traditional method)
tfidf = joblib.load(SAVE_DIR / 'tfidf_logreg.joblib')
tf_vectorizer = tfidf['vectorizer']
tfidf_model   = tfidf['model']

# fine-tuned RoBERTa (pretrained language model)
r_tokenizer   = AutoTokenizer.from_pretrained(str(SAVE_DIR / 'roberta_finetuned'))
roberta_model = AutoModelForSequenceClassification.from_pretrained(str(SAVE_DIR / 'roberta_finetuned')).to(device)
roberta_model.eval()

print("Loaded:", info['roberta_variant'], "and TF-IDF + LogReg. Labels:", EKMAN)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13182.21it/s]

Loaded: RoBERTa fine-tuned and TF-IDF + LogReg. Labels: ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']


## 4. Prediction function
Same logic as the main notebook: an emotion is predicted when its probability is above 0.5.

In [4]:
@torch.no_grad()
def predict_emotions(text):
    if not isinstance(text, str) or text.strip() == "":
        print("Please type a non-empty English sentence.")
        return

    # TF-IDF + LogReg (columns are in EKMAN order)
    tfidf_pred = tfidf_model.predict(tf_vectorizer.transform([text]))[0]
    tfidf_labels = [EKMAN[i] for i in range(7) if tfidf_pred[i] == 1]

    # fine-tuned RoBERTa
    enc = r_tokenizer([text], truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(device)
    probs = torch.sigmoid(roberta_model(**enc).logits)[0].cpu().numpy()
    roberta_labels = [EKMAN[i] for i in range(7) if probs[i] > THRESHOLD]

    print(text)
    print("  TF-IDF :", tfidf_labels or "(no emotion above 0.5)")
    print("  RoBERTa:", roberta_labels or f"(no emotion above 0.5, most likely: {EKMAN[probs.argmax()]})")
    print("  RoBERTa probabilities:", {EKMAN[i]: round(float(probs[i]), 2) for i in range(7)})

## 5. Try it
Type a sentence in the box and run the cell.

In [5]:
text = "I can't believe they cancelled the trip again."  #@param {type:"string"}
predict_emotions(text)

I can't believe they cancelled the trip again.
  TF-IDF : ['surprise']
  RoBERTa: ['surprise']
  RoBERTa probabilities: {'anger': 0.1, 'disgust': 0.0, 'fear': 0.01, 'joy': 0.02, 'sadness': 0.02, 'surprise': 0.89, 'neutral': 0.08}


### Demo sentences

In [6]:
examples = [
    "My hands are shaking as I open the results email.",
    "I finally got the internship, I can't stop smiling!",
    "Wow, thanks for nothing.",
    "",
]
for sentence in examples:
    predict_emotions(sentence)
    print()

My hands are shaking as I open the results email.
  TF-IDF : (no emotion above 0.5)
  RoBERTa: ['fear']
  RoBERTa probabilities: {'anger': 0.03, 'disgust': 0.01, 'fear': 0.7, 'joy': 0.02, 'sadness': 0.16, 'surprise': 0.05, 'neutral': 0.24}



I finally got the internship, I can't stop smiling!
  TF-IDF : (no emotion above 0.5)
  RoBERTa: ['joy']
  RoBERTa probabilities: {'anger': 0.0, 'disgust': 0.0, 'fear': 0.0, 'joy': 0.99, 'sadness': 0.0, 'surprise': 0.01, 'neutral': 0.01}

Wow, thanks for nothing.
  TF-IDF : ['joy']
  RoBERTa: ['joy']
  RoBERTa probabilities: {'anger': 0.01, 'disgust': 0.0, 'fear': 0.0, 'joy': 0.98, 'sadness': 0.0, 'surprise': 0.27, 'neutral': 0.01}

Please type a non-empty English sentence.

